# Structured Output with Strands Agents

## Overview

When you call a Strands agent, it returns free-form text by default. That's fine for chatbots, but when the agent's output needs to feed into downstream code — an API response, a database write, a UI component — you need structured, typed, validated data.

In this tutorial, we'll walk you through how to use structured output to get back validated Python objects instead of strings.

| Feature | Description |
|---------|-------------|
| **Flat Pydantic Models** | Define a model, pass it to the agent, and access typed results |
| **Complex Schemas** | Nested models, lists, optional fields, validators, and enums |
| **Validation & Self-Correction** | Automatic retry when validation fails, exception handling, custom forcing prompt |
| **Tools + Structured Output** | Combine tool use with structured results |

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude Sonnet 4.5 enabled on Amazon Bedrock, [guide](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access-modify.html)
* Familiarity with Strands Agents basics [(see Tutorial 01)](../01-first-agent/)

Let's now install the required packages

In [ ]:
%pip install -r requirements.txt -q

### Importing dependency packages

Now let's import the dependency packages

In [ ]:
from enum import Enum
from typing import List, Literal, Optional

from pydantic import BaseModel, Field, field_validator

from strands import Agent
from strands.types.exceptions import StructuredOutputException

In [ ]:
# Model used throughout this tutorial
MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

## 1. Your First Structured Output

Let's start with the simplest use case: you want the agent to return data in a specific shape instead of free-form text.

**The pattern:**
1. Define a Pydantic `BaseModel` with the fields you need
2. Pass it as `structured_output_model` when calling the agent
3. Access the typed result via `result.structured_output`

### Defining a Pydantic Model

This is a plain Pydantic model — the same kind you'd use for an API schema or database record. The `Field(description=...)` helps the LLM understand what each field expects.

In [ ]:
class MovieReview(BaseModel):
    """A structured movie review."""

    title: str = Field(description="The movie title")
    rating: int = Field(description="Rating from 1 to 10")
    summary: str = Field(description="A brief one-sentence summary of the review")
    recommend: bool = Field(description="Whether you would recommend this movie")

### Synchronous call

Let's pass `structured_output_model` when calling the agent. The result is a validated `MovieReview` instance — not a string, not a dict.

> **Note:** The exact field values in the output will vary between runs since the LLM generates content dynamically. The structure and types will always match your Pydantic model.

In [ ]:
agent = Agent(model=MODEL_ID)

result = agent("Review the movie Inception by Christopher Nolan", structured_output_model=MovieReview)

# Access the typed result
review = result.structured_output
print(f"Type: {type(review).__name__}")
print(f"Title: {review.title}")
print(f"Rating: {review.rating}/10")
print(f"Summary: {review.summary}")
print(f"Recommend: {review.recommend}")

> **Notice:** `result.structured_output` is a real `MovieReview` instance — `type(review).__name__` prints `MovieReview`, not `str` or `dict`. You get attribute access (`review.rating`) with the right Python types: `rating` is an `int` and `recommend` is a `bool`. No JSON parsing, no string wrangling.

### Asynchronous call

The same pattern works with `invoke_async()` for async workflows.

In [ ]:
async def get_review_async():
    agent = Agent(model=MODEL_ID)
    result = await agent.invoke_async(
        "Review the movie The Matrix",
        structured_output_model=MovieReview,
    )
    review = result.structured_output
    print(f"[Async] {review.title}: {review.rating}/10 — {review.summary}")


await get_review_async()

## 2. Complex Schemas

Real-world data isn't flat. Let's see how structured output handles the same Pydantic features you'd use in production: nested models, lists, optional fields, constrained values, and enums.

### Nested models and lists

Let's define sub-models and compose them. The SDK converts the full schema — including nested `$ref` types — into a tool specification the LLM can understand.

In [ ]:
class Address(BaseModel):
    """A physical address."""

    street: str = Field(description="Street address")
    city: str = Field(description="City name")
    state: str = Field(description="State or province")
    country: str = Field(description="Country")


class Skill(BaseModel):
    """A professional skill with proficiency level."""

    name: str = Field(description="Skill name")
    years_experience: int = Field(description="Years of experience", ge=0)


class Candidate(BaseModel):
    """A job candidate profile."""

    name: str = Field(description="Full name")
    address: Address = Field(description="Home address")
    skills: List[Skill] = Field(description="List of professional skills")
    bio: Optional[str] = Field(default=None, description="Short bio, if available")

In [ ]:
agent = Agent(model=MODEL_ID)

result = agent(
    "Create a profile for a fictional senior software engineer based in Seattle with 3 skills",
    structured_output_model=Candidate,
)

candidate = result.structured_output
print(f"Name: {candidate.name}")
print(f"Location: {candidate.address.city}, {candidate.address.state}")
print(f"Bio: {candidate.bio}")
print("Skills:")
for skill in candidate.skills:
    print(f"  - {skill.name}: {skill.years_experience} years")

> **Notice:** The nested structure is fully hydrated into typed objects. `candidate.address` is itself an `Address` instance (so `candidate.address.city` works), and `candidate.skills` is a `list` of `Skill` objects — not a list of dicts. If `bio` comes back as `None`, that's the `Optional[str]` field doing its job: the LLM is allowed to omit it.

### Field constraints and enums

We can use Pydantic's `Field` constraints (`ge`, `le`, `min_length`, etc.) and `Literal`/`Enum` types to restrict valid values. The LLM sees these constraints in the tool schema and respects them.

In [ ]:
class Priority(str, Enum):
    """Task priority levels."""

    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"


class TicketAnalysis(BaseModel):
    """Analysis of a customer support ticket."""

    category: str = Field(description="Issue category", min_length=1)
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="Customer sentiment")
    priority: Priority = Field(description="Ticket priority")
    confidence: float = Field(description="Confidence score", ge=0.0, le=1.0)
    summary: str = Field(description="Brief summary of the issue")

In [ ]:
agent = Agent(model=MODEL_ID)

ticket_text = """
I've been trying to reset my password for 3 days now and the reset email never arrives.
I've checked spam. This is blocking my entire team from accessing the dashboard.
We're paying for the enterprise plan and this level of service is unacceptable.
"""

result = agent(
    f"Analyze this support ticket:\n{ticket_text}",
    structured_output_model=TicketAnalysis,
)

analysis = result.structured_output
print(f"Category: {analysis.category}")
print(f"Sentiment: {analysis.sentiment}")
print(f"Priority: {analysis.priority.value}")
print(f"Confidence: {analysis.confidence:.0%}")
print(f"Summary: {analysis.summary}")

> **Notice a few things in this output:**
>
> - **`sentiment` is exactly one of three values.** It is `"positive"`, `"negative"`, or `"neutral"` — never "somewhat negative" or "frustrated". The `Literal[...]` type constrains the LLM to those exact strings.
> - **`confidence` is a float between 0 and 1.** The output shows `95%`, but the underlying value is `0.95` — the `:.0%` in the f-string is just display formatting. The `ge=0.0, le=1.0` constraints guarantee it stays in range.
> - **`priority` is a real enum member**, not a bare string. `analysis.priority` is `Priority.HIGH`; we print `analysis.priority.value` to get `"high"`.

## 3. Validation & Self-Correction

What happens when the LLM returns data that doesn't pass Pydantic validation?

The SDK handles this automatically:
1. The LLM calls the structured output tool with its data
2. Pydantic validates the data — if it fails, the SDK formats a per-field error message
3. The error is sent back to the LLM as a tool result
4. The LLM reads the error and self-corrects on the next attempt

This happens within the normal agent loop — no extra code needed.

### Triggering the retry loop

To *see* self-correction happen, we need a constraint the LLM is genuinely likely to get wrong on its first try. LLMs are notoriously bad at counting characters — so we'll require a field to be an **exact length**. The model will almost always overshoot or undershoot, the validator will reject it, and the error is sent back so the LLM can adjust.

In [ ]:
class SocialPost(BaseModel):
    """A social media post constrained to an exact character count."""

    platform: str = Field(description="The social platform the post is for")
    text: str = Field(
        description=(
            "Post text that is EXACTLY 40 characters long, including spaces "
            "and punctuation. Plain text only — no emojis."
        )
    )

    @field_validator("text")
    @classmethod
    def exactly_40_chars(cls, v: str) -> str:
        """Reject any text that is not exactly 40 characters."""
        if len(v) != 40:
            raise ValueError(
                f"text is {len(v)} characters but must be EXACTLY 40. "
                f"Rewrite it to be exactly 40 characters long."
            )
        return v

In [ ]:
agent = Agent(model=MODEL_ID)

# Counting characters exactly is hard for an LLM, so it will likely fail the
# validator on its first attempt, read the error, and try again until it fits.
result = agent(
    "Write a promotional post for our new coffee blend called 'Sunrise Roast'.",
    structured_output_model=SocialPost,
)

post = result.structured_output
print(f"Platform: {post.platform}")
print(f"Text:     {post.text!r}")
print(f"Length:   {len(post.text)} characters")

In [ ]:
# Inspect the conversation to see the self-correction trail.
# Each call to the SocialPost tool is one attempt; a rejected attempt is
# followed by a tool result containing the validation error.
attempt = 0
for message in agent.messages:
    for block in message.get("content", []):
        if "toolUse" in block and block["toolUse"]["name"] == "SocialPost":
            attempt += 1
            text = block["toolUse"]["input"].get("text", "")
            print(f"Attempt {attempt}: {text!r} (len={len(text)})")
        if "toolResult" in block and block["toolResult"].get("status") == "error":
            for item in block["toolResult"]["content"]:
                if "text" in item:
                    error_line = item["text"].splitlines()[-1].strip()
                    print(f"   rejected -> {error_line}")

print(f"\nTotal attempts: {attempt}")

> **Notice the self-correction loop in action.** The early attempt(s) came back with the wrong length (e.g., 37 or 39 characters). Pydantic's `@field_validator` raised a `ValueError`, the SDK packaged that error as a tool result and sent it back to the LLM, and the LLM rewrote the text until it hit exactly 40 — all inside a single `agent(...)` call, with no extra code from you.
>
> Because the LLM is non-deterministic, the number of attempts varies between runs. Occasionally it nails 40 characters on the first try (a single attempt); usually it takes two or more.

### Handling StructuredOutputException

If the LLM fails to produce valid output even after the SDK forces it, a `StructuredOutputException` is raised. Always wrap structured output calls in a try/except for production code.

In [ ]:
agent = Agent(model=MODEL_ID)

try:
    result = agent(
        "Write a launch announcement post for our mobile app.",
        structured_output_model=SocialPost,
    )
    post = result.structured_output
    print(f"Success: {post.text!r} ({len(post.text)} chars)")
except StructuredOutputException as e:
    print(f"Structured output failed: {e}")
    print("Fallback: use the raw text response or retry with a different prompt")

### Custom forcing prompt

When the LLM doesn't call the structured output tool on its own, the SDK sends a forcing prompt to nudge it. The default is:

> *"You must format the previous response as structured output."*

You can customize this with `structured_output_prompt` to give the LLM more specific guidance for your schema.

In [ ]:
agent = Agent(model=MODEL_ID)

result = agent(
    "Write a post announcing a weekend sale.",
    structured_output_model=SocialPost,
    structured_output_prompt=(
        "Format your response using the structured output tool. "
        "The 'text' field MUST be exactly 40 characters long — count carefully, "
        "including spaces and punctuation."
    ),
)

post = result.structured_output
print(f"Platform: {post.platform}")
print(f"Text:     {post.text!r}")
print(f"Length:   {len(post.text)} characters")

## 4. Combining Tools with Structured Output

In practice, agents don't just format text — they use tools to gather information first, then structure the result. The structured output tool coexists with regular tools. The LLM uses regular tools to do work, then calls the structured output tool last to return the final answer.

Let's define a small `calculator` tool so the agent can perform calculations before returning a structured result.

In [ ]:
import operator
from typing import Literal

from strands import tool

_OPS = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
    "**": operator.pow,
}


@tool
def calculator(a: float, b: float, op: Literal["+", "-", "*", "/", "**"]) -> float:
    """Apply an arithmetic operator to two numbers.

    Args:
        a: Left operand.
        b: Right operand.
        op: One of "+", "-", "*", "/", "**".
    """
    return _OPS[op](a, b)


class InvestmentAnalysis(BaseModel):
    """Analysis of an investment calculation."""

    initial_amount: float = Field(description="Initial investment amount in dollars")
    annual_rate: float = Field(description="Annual interest rate as a percentage")
    years: int = Field(description="Investment period in years")
    final_amount: float = Field(description="Calculated final amount after compound interest")
    total_return: float = Field(description="Total return as a percentage")
    recommendation: str = Field(description="Brief investment recommendation")

In [ ]:
agent = Agent(
    model=MODEL_ID,
    tools=[calculator],
    system_prompt="You are a financial analyst. Use the calculator tool for all math operations.",
)

result = agent(
    "If I invest $10,000 at 7% annual compound interest for 15 years, what will I have? Analyze this investment.",
    structured_output_model=InvestmentAnalysis,
)

analysis = result.structured_output
print(f"Initial: ${analysis.initial_amount:,.2f}")
print(f"Rate: {analysis.annual_rate}%")
print(f"Period: {analysis.years} years")
print(f"Final: ${analysis.final_amount:,.2f}")
print(f"Return: {analysis.total_return:.1f}%")
print(f"Recommendation: {analysis.recommendation}")

The agent worked through the compound-interest formula step by step, chaining several `calculator` calls (one per arithmetic operation) before calling the structured output tool to return the result as a validated `InvestmentAnalysis` object. The multi-call chain is the tool loop working as designed: each intermediate value flows back through the model into the next call, and the final numbers land in a typed object rather than prose.

> **Notice:** `final_amount` and `total_return` come back as real `float`s (not text), so you can use them directly in further calculations, store them in a database, or return them from an API — no parsing required.

## Key Takeaways

You've learned how to get reliable, typed data from Strands agents:

1. **Basic structured output** — Define a Pydantic model, pass it to the agent, get a validated object back
2. **Complex schemas** — Nested models, lists, optional fields, field constraints, and enums all work
3. **Validation & self-correction** — When output fails validation, the SDK sends the error back and the LLM retries automatically; you saw the retry trail in `agent.messages`. Catch `StructuredOutputException` as a fallback.
4. **Tools + structured output** — Agents use tools to gather data, then return structured results

### Tips for production use

* Use `structured_output_model=` (not the deprecated `agent.structured_output()` method)
* Add `Field(description=...)` to help the LLM understand your schema
* Always wrap production calls in `try/except StructuredOutputException`
* Use `structured_output_prompt` to improve success rates for strict schemas
* Structured output works with both sync (`agent()`) and async (`invoke_async()`)